In [1]:
# import pandas as pd
# from glob import glob

import pandas as pd 
import numpy as np 
from glob import glob
import os
import geopandas as gpd

import csv
import json

from ast import literal_eval
from shapely import wkt

/usr/local/lib/python3.8/dist-packages/geopandas/_compat.py:123: UserWarning: The Shapely GEOS version (3.11.2-CAPI-1.17.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
/tmp/tmp.lsKwyw7YSQ/ipykernel_1229752/3428607455.py:8: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration

## Prepare data 
- mappings Events <-> Cams (used for buffer times, etc)
- general geometries <-> Cams (no immediate use, but if we add new events in future)

#### Do not need to run again! Saved out as csvs

In [ ]:
# from /home/csutter/DRIVE-clean/weather_events/notebooks/stats_buffer_modelpred.py
# But want to run longer out buffer times so need to add more

In [2]:
######### CONFIG

alldirs_data_preds = glob("/home/csutter/DRIVE-clean/operational_runs_wMRMS/data_6_ensembling") #HERE!! 
# data_6_ensembling
# data_odm_3_ensembling

csv_path = "/home/csutter/DRIVE-clean/weather_events/models/stats_events_modelpred/stats_buffer_rsc.csv"# HERE!!!


alldirs_data_preds[0:4]

['/home/csutter/DRIVE-clean/operational_runs_wMRMS/data_6_ensembling']

In [3]:

########## Grab events
#### 1 - NCEI data
d_readin = gpd.read_file("/home/csutter/DRIVE-clean/weather_events/data/ncei_events/ncei_ny_events_clean.gpkg")

# add timedelta duration col back (using duration_sec col)
d_readin["duration"] = pd.to_timedelta(d_readin["duration_sec"], unit="s")



In [4]:
d_readin.head(3)

,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,...,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration
0,14,202202,3,900,202202,4,1600,164922,995747,NEW YORK,...,CSV,031,031,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.56300 44.50731, -73.55910 44.479...",1 days 07:00:00
1,15,202202,3,900,202202,4,1600,164922,995749,NEW YORK,...,CSV,030,030,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.98640 44.70781, -73.96619 44.709...",1 days 07:00:00
2,16,202202,3,900,202202,4,1600,164922,995750,NEW YORK,...,CSV,027,027,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-74.66310 44.99891, -74.66100 44.999...",1 days 07:00:00


In [5]:
display(d_readin[d_readin["EVENT_ID"]==995747] )
# event is unique to one event at one zone
# episode connects different zones (but from the same weather system)

print(len(np.unique(d_readin["EVENT_ID"]))) # 7800 different events we have in our ncei events database

,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,...,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration
0,14,202202,3,900,202202,4,1600,164922,995747,NEW YORK,...,CSV,031,031,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.56300 44.50731, -73.55910 44.479...",1 days 07:00:00


7802


In [6]:
#### 2 - Cam lat and lons 
cams = pd.read_csv("/home/csutter/DRIVE/site_analysis/_reference/511NY_API_GetCameras_response.csv")

cams = cams[((cams["Disabled"]==False)&(cams["Blocked"]==False))]

# display(cams.head(3))
print(len(cams))

cams_gdf = gpd.GeoDataFrame(
    cams, 
    geometry=gpd.points_from_xy(cams.Longitude, cams.Latitude),
    crs="EPSG:4326" # Start with standard GPS coordinates
)

cams_gdf = cams_gdf.to_crs(d_readin.crs) # match the CRS to be exactly the system being used in the events dataset

# display(cams_gdf.head(3))
print(len(cams_gdf))

# Perform the spatial join
# This gives us a dataframe mapping EVERY event to EVERY camera within that event's zone -- new row for each cam | event. However, these events are more than our events of interest, so will end up subsetting this / cleaning the data below.
events_camloc = gpd.sjoin(
    cams_gdf, 
    d_readin[["EVENT_ID","geometry"]], 
    how="inner",
    predicate="within" # Check if the point is WITHIN the polygon
)

# display(events_camloc.head(3)) 
print(len(events_camloc)) # 2300 cams, 7802 events, when we tie the cams to events, we get multiple cams for one event, and likewise, multple events for a single cam -- final amount of joined dataset is 145,470. Not every zone has cams, and some have more, some have less

display(events_camloc[events_camloc["ID"]=="Skyline-1867"].head(3))

2371
2371
145470


,Unnamed: 0,Latitude,Longitude,ID,Name,DirectionOfTravel,RoadwayName,Url,VideoUrl,Disabled,Blocked,geometry,index_right,EVENT_ID
95,95,40.767013,-73.696306,Skyline-1867,I-495 West of New Hyde Park Rd,Eastbound,I-495,https://511ny.org/map/Cctv/1867--1,https://s52.nysdot.skyvdn.com:443/rtplive/R10_...,False,False,POINT (-73.69631 40.76701),204,996924
95,95,40.767013,-73.696306,Skyline-1867,I-495 West of New Hyde Park Rd,Eastbound,I-495,https://511ny.org/map/Cctv/1867--1,https://s52.nysdot.skyvdn.com:443/rtplive/R10_...,False,False,POINT (-73.69631 40.76701),236,994061
95,95,40.767013,-73.696306,Skyline-1867,I-495 West of New Hyde Park Rd,Eastbound,I-495,https://511ny.org/map/Cctv/1867--1,https://s52.nysdot.skyvdn.com:443/rtplive/R10_...,False,False,POINT (-73.69631 40.76701),323,999831


In [7]:
#### Grab SUBSET of events data that we care about (after reading these in, will subset the df we made above to just these events that we care about)

events_ofinterest_paths = [
    "/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/blizzard_allyrs_ceilfloor5min_nobuffer_freq5min.csv",
    "/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/heavyrain_2425_ceilfloor15min_nobuffer_freq15min.csv",
    "/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/heavysnow_2025_ceilfloor5min_nobuffer_freq5min.csv",
    "/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/heavysnow_2425_ceilfloor15min_nobuffer_freq15min.csv",
    "/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/lakeeffect_2025_ceilfloor15min_nobuffer_freq15min.csv",
    "/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/lakeeffect_2025_ceilfloor30min_nobuffer_freq30min.csv",
    "/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/lakeeffect_2425_ceilfloor15min_nobuffer_freq15min.csv",
    "/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/winterstorm_2425_ceilfloor15min_nobuffer_freq15min.csv",
    "/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/winterweather_2425_ceilfloor15min_nobuffer_freq15min.csv"
]

gdfs_list = []

for ev in events_ofinterest_paths:
    events = pd.read_csv(ev)
    events['geometry'] = events['geometry'].apply(wkt.loads)
    events_gdf = gpd.GeoDataFrame(events, geometry='geometry', crs="EPSG:4269")
    events_gdf = events_gdf.to_crs(d_readin.crs)
    gdfs_list.append(events_gdf)

eventsofint = pd.concat(gdfs_list, ignore_index=True)

print(len(eventsofint)) # these are just the winter weather events and others that we care about, not all events (from the original d_readin)

1056


In [14]:
#### Keep just events of interest (inner join on EVENT ID)
# This the main df to keep! Has all events of interest and their corresponding cam info, should be unique by event | cam, which are EVENT_ID and ID

eventsofint_camloc = eventsofint.merge(events_camloc[['Latitude', 'Longitude', 'ID', 'Name','DirectionOfTravel', 'RoadwayName', 'Url', 'VideoUrl', 'Disabled','Blocked','EVENT_ID']], how="inner", on="EVENT_ID")

display(eventsofint_camloc.head(3))

print(len(eventsofint))
print(len(eventsofint_camloc)) ## expanded bc adding rows for every cam for every event (as opposed to one row per event)

print(len(np.unique(eventsofint["EVENT_ID"]))) 
print(len(np.unique(eventsofint_camloc["EVENT_ID"]))) # reflecting that not every event (which is at the geometry level has cams in it). These are the number of events that have cams in them. 

,Unnamed: 0.1,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,...,Latitude,Longitude,ID,Name,DirectionOfTravel,RoadwayName,Url,VideoUrl,Disabled,Blocked
0,118,4768,202201,29,700,202201,29,1330,165058,996866,...,40.813286,-73.108309,Skyline-1879,I-495 at Hawkins Ave,Eastbound,I-495,https://511ny.org/map/Cctv/1879--1,https://s52.nysdot.skyvdn.com:443/rtplive/R10_...,False,False
1,118,4768,202201,29,700,202201,29,1330,165058,996866,...,40.815508,-73.084111,Skyline-1880,I-495 at Patchogue-Holbrook Rd,Unknown,I-495,https://511ny.org/map/Cctv/1880--1,https://s52.nysdot.skyvdn.com:443/rtplive/R10_...,False,False
2,118,4768,202201,29,700,202201,29,1330,165058,996866,...,40.816612,-73.077754,Skyline-1881,I-495 at Holbrook Rd,Unknown,I-495,https://511ny.org/map/Cctv/1881--1,https://s52.nysdot.skyvdn.com:443/rtplive/R10_...,False,False


1056
841
13055
495


In [15]:
# SAVE OUT
eventsofint_camloc.to_csv("/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest_with_cams/events_ofint_withcams.csv")

In [12]:
# Makes another reference dataframe in case needed in future 
# This is just to save out the bare geometry <-> cam mapping. 
# E.g. county zone | wfo | geometry | ID(which is cam ID) -- that way, should we have new events in the future, can just use this mapping to colocate the cams rather than running all those above. However, for case study and analyses for dissertation, just use the DF above as it has events of interest and everything else we care about. 

locs = eventsofint_camloc[['geometry','CZ_FIPS', 'CZ_NAME', 'WFO','CZ_FIPS_FORMAT', 'ZONE', 'FIPS', 'FIPS_FORMAT','Latitude', 'Longitude', 'ID', 'Name','DirectionOfTravel', 'RoadwayName', 'Url', 'VideoUrl', 'Disabled','Blocked']]
locsunique = locs.drop_duplicates()
locsunique['ID'] = locsunique['ID'].str.replace('-', '_')

locsunique_bare = locsunique[["geometry","CZ_NAME","WFO", "ID"]]

# display(locs.head(3))
# display(locsunique.head(3))
# display(locsunique_bare.head(3))

/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [16]:
locsunique.to_csv("/home/csutter/DRIVE-clean/weather_events/data/ncei_geom_to_cam_mapping/ncei_geom_to_cam.csv")